In [24]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [25]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [26]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [27]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [28]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [29]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [30]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [31]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 362,
 'tn': 2637,
 'fp': 0,
 'fn': 1,
 'misclassification_rate': 0.0003333333333333333,
 'false_positive_rate': 0.0,
 'false_negative_rate': 0.0027548209366391185}

### Check results on the test set (new data not yet seen by the model)

In [32]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 31,
 'tn': 837,
 'fp': 37,
 'fn': 95,
 'misclassification_rate': 0.132,
 'false_positive_rate': 0.04233409610983982,
 'false_negative_rate': 0.753968253968254}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

The misclassification rate is relatively low, so the model must have the ability correctly predict a bot a majority of the time.

### What are potential ramifications of false positives from the model?

False positives can lead to people being falsely accused of being bots.

### What are potential ramifications of false negatives from the model?

False negatives means bots can be mistaken for people, leading others to trust a bot.